In [11]:
import os
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    precision_recall_fscore_support,
)

from imblearn.under_sampling import RandomUnderSampler
import joblib


In [13]:
paysim_path = "/Users/nitishbhattad/Desktop/kafka-fraud-detection/data/paysim.csv"

df_raw = pd.read_csv(paysim_path)
df_raw.columns = df_raw.columns.str.strip()  # safety

print("Raw shape:", df_raw.shape)
print("Columns:", df_raw.columns.tolist())
print(df_raw[["isFraud"]].value_counts(normalize=True).rename("ratio"))



Raw shape: (6362620, 11)
Columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']
isFraud
0          0.998709
1          0.001291
Name: ratio, dtype: float64


In [15]:
features = [
    "type",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",   # note: this matches your file
    "oldbalanceDest",
    "newbalanceDest",
]
target = "isFraud"

df_model = df_raw[features + [target]].copy()
print("Model df shape:", df_model.shape)

Model df shape: (6362620, 7)


In [5]:
df_model = df[features + [target]].copy()
print(df_model.shape)
df_model.head()


(6362620, 7)


,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud
0,PAYMENT,9839.64,170136.0,160296.36,0.0,0.0,0
1,PAYMENT,1864.28,21249.0,19384.72,0.0,0.0,0
2,TRANSFER,181.00,181.0,0.00,0.0,0.0,1
3,CASH_OUT,181.00,181.0,0.00,21182.0,0.0,1
4,PAYMENT,11668.14,41554.0,29885.86,0.0,0.0,0


In [17]:
features = [
    "type",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",   # note: this matches your file
    "oldbalanceDest",
    "newbalanceDest",
]
target = "isFraud"

df_model = df_raw[features + [target]].copy()
print("Model df shape:", df_model.shape)

Model df shape: (6362620, 7)


In [19]:
df_model_sampled = df_model.sample(n=200_000, random_state=42)
print("Sampled label distribution:")
print(df_model_sampled[target].value_counts(), df_model_sampled.shape)

data = df_model_sampled  # or df_model for full dataset


Sampled label distribution:
isFraud
0    199732
1       268
Name: count, dtype: int64 (200000, 7)


In [21]:
X = data[features].copy()
y = data[target].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train fraud ratio:", y_train.mean(), "Test fraud ratio:", y_test.mean())


Train shape: (160000, 6) Test shape: (40000, 6)
Train fraud ratio: 0.0013375 Test fraud ratio: 0.00135


In [23]:
cat_features = ["type"]
num_features = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
]

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
        ("num", StandardScaler(), num_features),
    ],
    remainder="drop",
)

In [25]:
rus = RandomUnderSampler(
    sampling_strategy=0.2,  # keep majority ≈5x minority
    random_state=42,
)

X_train_res, y_train_res = rus.fit_resample(X_train, y_train)

print("\nBefore resampling:")
print(y_train.value_counts())
print("\nAfter resampling:")
print(y_train_res.value_counts())


Before resampling:
isFraud
0    159786
1       214
Name: count, dtype: int64

After resampling:
isFraud
0    1070
1     214
Name: count, dtype: int64


In [27]:
rf_clf_resampled = Pipeline(
    steps=[
        ("preprocess", preprocess),
        (
            "clf",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=None,
                n_jobs=-1,
                random_state=42,
                class_weight=None,  # we already balanced via sampling
            ),
        ),
    ]
)

rf_clf_resampled.fit(X_train_res, y_train_res)

y_proba_rf_res = rf_clf_resampled.predict_proba(X_test)[:, 1]

print("\n=== Random Forest (resampled) – AUC ===")
print("ROC-AUC:", roc_auc_score(y_test, y_proba_rf_res))



=== Random Forest (resampled) – AUC ===
ROC-AUC: 0.998851226934139


In [29]:
thresholds = np.linspace(0.1, 0.9, 9)
rows = []

for thr in thresholds:
    y_pred_thr = (y_proba_rf_res >= thr).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, y_pred_thr, average="binary", zero_division=0
    )
    rows.append(
        {
            "threshold": thr,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        }
    )

thr_df = pd.DataFrame(rows)
print("\n=== Threshold scan (fraud class) ===")
print(thr_df)


=== Threshold scan (fraud class) ===
   threshold  precision    recall        f1
0        0.1   0.019054  1.000000  0.037396
1        0.2   0.035952  1.000000  0.069409
2        0.3   0.058306  0.981481  0.110073
3        0.4   0.090121  0.962963  0.164818
4        0.5   0.138667  0.962963  0.242424
5        0.6   0.182143  0.944444  0.305389
6        0.7   0.260417  0.925926  0.406504
7        0.8   0.393443  0.888889  0.545455
8        0.9   0.698413  0.814815  0.752137


In [31]:
best_row = thr_df.sort_values("f1", ascending=False).iloc[0]
best_thr = float(best_row["threshold"])

print("\nBest threshold by F1:")
print(best_row)


Best threshold by F1:
threshold    0.900000
precision    0.698413
recall       0.814815
f1           0.752137
Name: 8, dtype: float64


In [33]:
y_pred_best = (y_proba_rf_res >= best_thr).astype(int)

print("\n=== Final model at chosen threshold ===")
print(classification_report(y_test, y_pred_best))


=== Final model at chosen threshold ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     39946
           1       0.70      0.81      0.75        54

    accuracy                           1.00     40000
   macro avg       0.85      0.91      0.88     40000
weighted avg       1.00      1.00      1.00     40000



In [35]:
model_dir = "/Users/nitishbhattad/Desktop/kafka-fraud-detection/models"
os.makedirs(model_dir, exist_ok=True)

model_path = os.path.join(model_dir, "fraud_model_paysim_rf.pkl")
joblib.dump(rf_clf_resampled, model_path)

config = {
    "features": features,
    "categorical": cat_features,
    "numerical": num_features,
    "threshold": best_thr,
}

config_path = os.path.join(model_dir, "fraud_model_paysim_config.json")
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print("\nSaved model to:", model_path)
print("Saved config to:", config_path)



Saved model to: /Users/nitishbhattad/Desktop/kafka-fraud-detection/models/fraud_model_paysim_rf.pkl
Saved config to: /Users/nitishbhattad/Desktop/kafka-fraud-detection/models/fraud_model_paysim_config.json
